# Plastic RNN vs. Vanilla RNN on the Frequency-Encoded Adding Problem

The leaky RNN in `03_leaky_rnn.ipynb` gave each **node** a memory. This
notebook does the opposite: the nodes stay **static** (memoryless), and the
**weights** become dynamic. The recurrent connectivity is a gradient-trained
backbone $W_{hh}$ plus a fast plastic component $A_t$ that evolves *within
each sequence* under the associative Hebbian rule of Wakhloo, Clark & Abbott
(2025) (Eq. 3 in `papers/wakhloo2025/persistent_oscillations.py`):

$$p\,\frac{dA_{ij}}{dt} = -A_{ij} + \frac{k}{N}\,\phi(x_i)\,\phi(x_j)$$

Euler-discretizing the paper's neuron equation with $\Delta t = \tau$ removes
the leak and gives exactly the vanilla update, which is what "static nodes"
means here. Discretizing the synapse equation with the same step gives

$$h_t = \tanh\!\big(W_{xh}x_t + b + (W_{hh} + A_{t-1})\,h_{t-1}\big), \qquad
A_t = \Big(1-\tfrac{1}{p}\Big)A_{t-1} + \frac{k}{pN}\,h_t h_t^\top, \qquad A_0 = 0.$$

$A$ is a per-sequence *state*, reset to zero for every sequence as in the
paper; the optimizer never touches it. Backprop flows through $A$'s
trajectory, so the backbone can learn to *use* plasticity. We use the
paper's values $k = 2.8$ and $p = 25$ (synapses 25× slower than neurons,
i.e. 25 timesteps). With $k=0$ the model is exactly the vanilla RNN. The
model is in `biornn/models/plastic_rnn.py`.

Tasks, seeds, optimizer and hyperparameters match `03_leaky_rnn.ipynb`
(hidden size 64, Adam at `lr=1e-2`, 1200 epochs, batch 128, seed 42,
`window_len=5`).

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt

from biornn.models.rnn import AddingRNN
from biornn.models.plastic_rnn import PlasticAddingRNN
from biornn.tasks import frequency_adding_problem
from biornn.device import get_device
from biornn.visualize import plot_comparison_bars

K, P = 2.8, 25.0  # plasticity strength and synaptic timescale from Wakhloo et al. (2025)

# Training loops run on DEVICE. `BIORNN_DEVICE=cpu` overrides it; see biornn/device.py
# for why the CPU can be faster for these small models.
DEVICE = get_device()
print('training on', DEVICE)

## 1. Sanity check: k = 0 is the vanilla RNN

With the same seed both models get identical $W_{xh}$, $W_{hh}$ and readout.
With $k=0$ the plastic component stays at zero, so the outputs should match
to the bit. With the paper's $k$ they shouldn't.

In [ ]:
x, _ = frequency_adding_problem(16, 40, rng=np.random.default_rng(0))
torch.manual_seed(0); vanilla = AddingRNN(hidden_size=64)
torch.manual_seed(0); plastic_k0 = PlasticAddingRNN(hidden_size=64, k=0.0, p=P)
torch.manual_seed(0); plastic = PlasticAddingRNN(hidden_size=64, k=K, p=P)
with torch.no_grad():
    print('max |vanilla - plastic(k=0)|  :', (vanilla(x) - plastic_k0(x)).abs().max().item())
    print(f'max |vanilla - plastic(k={K})|:', (vanilla(x) - plastic(x)).abs().max().item())

## 2. What the plastic weights do during one sequence (untrained network)

Run one `seq_len=100` sequence through the untrained plastic network and
track $A_t$: its size $\|A_t\|_F$, and the largest eigenvalue of the total
connectivity $W_{hh} + A_t$. In the paper, plasticity pulls eigenvalues out
of the random bulk, and that is what sustains activity once the input stops.

In [ ]:
torch.manual_seed(0)
model = PlasticAddingRNN(hidden_size=64, k=K, p=P)
x, y = frequency_adding_problem(1, 100, window_len=5, rng=np.random.default_rng(3))
with torch.no_grad():
    _, hs, As = model(x, return_hidden_states=True, return_plastic=True)
W = model.cell.W_hh.weight.detach()
A_norm = [A[0].norm().item() for A in As]
top_eig = [torch.linalg.eigvals(W + A[0]).abs().max().item() for A in As]
rate = [h[0].pow(2).mean().item() for h in hs]
print(f'spectral radius of W_hh alone: {torch.linalg.eigvals(W).abs().max().item():.3f}')
print(f'||A||_F  at t=10, 50, 99: {A_norm[10]:.3f}, {A_norm[50]:.3f}, {A_norm[99]:.3f}')
print(f'max |eig(W_hh + A_t)| at t=10, 50, 99: {top_eig[10]:.3f}, {top_eig[50]:.3f}, {top_eig[99]:.3f}')

fig, axes = plt.subplots(1, 4, figsize=(17, 3.8))
axes[0].plot(x[0, :, 0], color='k', lw=1, label='signal')
axes[0].fill_between(range(100), -1, 1, where=x[0, :, 1] > 0, color='#ffb703', alpha=0.4, label='window')
axes[0].set(title='input', xlabel='t'); axes[0].legend(fontsize=8)
axes[1].plot(rate, color='#8338ec'); axes[1].set(title=r'mean squared rate  $\langle h_t^2 \rangle$', xlabel='t')
axes[2].plot(A_norm, color='#219ebc', label=r'$\|A_t\|_F$')
axes[2].plot(top_eig, color='#c1121f', label=r'max $|\lambda(W_{hh}+A_t)|$')
axes[2].axhline(1.0, color='gray', ls='--', lw=1); axes[2].set(title='plastic component', xlabel='t'); axes[2].legend(fontsize=8)
im = axes[3].imshow(As[-1][0].numpy(), cmap='RdBu_r', vmin=-As[-1][0].abs().max(), vmax=As[-1][0].abs().max())
axes[3].set(title=r'$A_{T}$ (post $\times$ pre)'); fig.colorbar(im, ax=axes[3], fraction=0.046)
fig.tight_layout(); plt.show()

Three things stand out, and the third is the important one:

- **Plasticity does move eigenvalues out of the bulk, as in the paper.**
  $W_{hh}$ alone has spectral radius 0.61. The largest eigenvalue of
  $W_{hh}+A_t$ crosses 1 around $t\approx45$ and reaches 2.35 by $t=99$.
- **But $A$ grows almost regardless of the input.** $\|A_t\|_F$ rises
  roughly linearly from $t=0$ (0.21 at $t=10$, 0.99 at $t=50$, 2.17 at
  $t=99$). The two stimulus windows leave only small kinks. Mean squared
  rate climbs from about 0.25 to 0.93, so the units are heading into
  saturation.
- **$A_T$ is essentially rank 1:** a fixed sign pattern times itself. The
  Hebbian term $h_t h_t^\top$ reinforces whatever activity pattern the
  network already has. Here that pattern is set mostly by the constant bias,
  not by the cosine snippets, so it's a positive-feedback loop that writes
  the *same* pattern into $A$ at every step.

In the paper, the oscillating drive keeps changing the activity pattern, so
$A$ ends up encoding the stimulus. Here the stimulus is on for only 10 of
100 steps, and the self-reinforcing mode dominates. This is for an
untrained network; Section 4 shows whether training can escape it.

## 3. Gradient flow at initialization

On one batch of the task at `seq_len=100`, backpropagate the MSE and measure
$\|\partial L/\partial h_t\|$ at every step, for vanilla and plastic models
with identical backbone weights. The plastic model has an extra backward
path through $A$ that decays like $(1-1/p)$ per step instead of through
repeated multiplication by the recurrent Jacobian.

In [ ]:
def grad_vs_lag(model, seq_len=100, seed=0):
    x, y = frequency_adding_problem(128, seq_len, window_len=5, rng=np.random.default_rng(seed))
    pred, hs = model(x, return_hidden_states=True)[:2]
    nn.functional.mse_loss(pred, y).backward()
    return np.array([h.grad.norm().item() for h in hs])[::-1]  # index by lag T - t

torch.manual_seed(0); g_vanilla = grad_vs_lag(AddingRNN(hidden_size=64))
torch.manual_seed(0); g_plastic = grad_vs_lag(PlasticAddingRNN(hidden_size=64, k=K, p=P))
for lag in [10, 30, 60, 90]:
    print(f'lag {lag:>2}:  vanilla {g_vanilla[lag]:.2e}   plastic {g_plastic[lag]:.2e}')

fig, ax = plt.subplots(figsize=(7, 4.5))
ax.plot(g_vanilla, color='#c1121f', label='vanilla (k=0)')
ax.plot(g_plastic, color='#219ebc', label=f'plastic (k={K}, p={P:g})')
ax.set_yscale('log'); ax.set(xlabel='time lag  (T - t)', ylabel=r'$\|\partial L / \partial h_t\|$',
                             title='Backpropagated gradient at initialization, seq_len=100')
ax.legend(); fig.tight_layout(); plt.show()

The backward pass is transformed. The vanilla gradient vanishes
geometrically ($10^{-4}$ at lag 10, $10^{-20}$ at lag 60, exactly 0 by lag
90). The plastic gradient **doesn't decay with lag at all**: it stays at
about $10^{-2}$ from lag 30 to lag 90, and is actually slightly *larger* at
long lags than at lag 10. The slow $(1-1/p)$ path through $A$ carries error
signal back across the whole sequence.

On gradient flow alone, the plastic network should have no trouble with
long sequences. Section 4 shows that isn't enough.

## 4. Training sweep: vanilla vs. plastic

Each pair differs only in $k$ (0 vs. 2.8), with identical initial weights
and data stream. Unlike the earlier notebooks, which reported the loss of
the last training batch, we report MSE on a held-out set of 1024 sequences,
which is less noisy. The predict-the-mean baseline is
$\mathrm{Var}(f_1+f_2) = 2/12 \approx 0.167$.

In [ ]:
def train_adding(seq_len, k, seed=42, epochs=1200, hidden_size=64, lr=1e-2, window_len=5, learn_k=False):
    torch.manual_seed(seed)
    rng = np.random.default_rng(seed)
    model = PlasticAddingRNN(hidden_size=hidden_size, k=k, p=P, learn_k=learn_k).to(DEVICE)
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    losses = []
    for epoch in range(epochs):
        x, y = frequency_adding_problem(128, seq_len, window_len=window_len, rng=rng, device=DEVICE)
        opt.zero_grad()
        loss = nn.functional.mse_loss(model(x), y)
        loss.backward()
        opt.step()
        losses.append(loss.item())
    return model, losses

def test_mse(model, seq_len, window_len=5, seed=12345):
    x, y = frequency_adding_problem(1024, seq_len, window_len=window_len, rng=np.random.default_rng(seed), device=DEVICE)
    with torch.no_grad():
        return nn.functional.mse_loss(model(x), y).item()

seq_lens = [40, 60, 100, 150]
models, curves = {}, {}
vanilla_test, plastic_test = [], []
for seq_len in seq_lens:
    for k in [0.0, K]:
        models[seq_len, k], curves[seq_len, k] = train_adding(seq_len, k)
    vanilla_test.append(test_mse(models[seq_len, 0.0], seq_len))
    plastic_test.append(test_mse(models[seq_len, K], seq_len))
    print(f'seq_len={seq_len:>4}  vanilla={vanilla_test[-1]:.4f}  plastic={plastic_test[-1]:.4f}')

fig = plot_comparison_bars(
    seq_lens, vanilla_test, plastic_test,
    ylabel='test MSE', title='Frequency-Encoded Adding Problem: vanilla vs. plastic RNN',
    baseline_label='vanilla RNN (k=0)', fixed_label=f'plastic RNN (k={K}, p={P:g})',
    reference=2/12, reference_label='predict-the-mean baseline',
)
plt.show()

Comparing different hidden layer size:

In [ ]:
seq_lens = [40, 60]
hidden_sizes = [2**n for n in range(3, 7)]            # 8, 16, 32, 64
size_models, size_curves, size_test = {}, {}, {}
for seq_len in seq_lens:
    for hidden in hidden_sizes:
        for k in [0.0, K]:                            # k=0 is the vanilla RNN
            m, c = train_adding(seq_len,k, epochs=300, hidden_size=hidden)
            size_models[seq_len, hidden, k], size_curves[seq_len, hidden, k] = m, c
            size_test[seq_len, hidden, k] = test_mse(m, seq_len)
        print(f'seq_len={seq_len:>3}  hidden={hidden:>3}  '
              f'vanilla={size_test[seq_len, hidden, 0.0]:.4f}  plastic={size_test[seq_len, hidden, K]:.4f}')

fig, ax = plt.subplots(figsize=(7, 4.5))
for seq_len, color in zip(seq_lens, ['#219ebc', '#fb8500']):
    ax.plot(hidden_sizes, [size_test[seq_len, h, K] for h in hidden_sizes], 'o-', color=color,
            label=f'plastic, seq_len={seq_len}')
    ax.plot(hidden_sizes, [size_test[seq_len, h, 0.0] for h in hidden_sizes], 's--', color=color, alpha=0.6,
            label=f'vanilla, seq_len={seq_len}')
ax.axhline(2/12, color='gray', ls='--', lw=1, label='predict-the-mean baseline')
ax.set(xscale='log', xticks=hidden_sizes, xticklabels=hidden_sizes, yscale='log',
       xlabel='hidden size', ylabel='test MSE', title=f'Effect of hidden size (plastic: k={K}, p={P:g})')
ax.legend(fontsize=8)
fig.tight_layout()
plt.show()


In [ ]:
fig, axes = plt.subplots(1, len(seq_lens), figsize=(18, 3.5), sharey=True)
smooth = lambda v, w=25: np.convolve(v, np.ones(w) / w, mode='valid')
for ax, seq_len in zip(axes, seq_lens):
    ax.plot(smooth(curves[seq_len, 0.0]), color='#c1121f', label='vanilla')
    ax.plot(smooth(curves[seq_len, K]), color='#219ebc', label='plastic')
    ax.axhline(2/12, color='gray', ls='--', lw=1)
    ax.set(yscale='log', title=f'seq_len={seq_len}', xlabel='epoch')
axes[0].set_ylabel('train MSE (25-epoch mean)'); axes[0].legend()
fig.tight_layout(); plt.show()

- **seq_len=20:** both solve it. Plastic is a bit worse (0.0022 vs 0.0004),
  the same small cost at the easiest length that the leak showed.
- **seq_len=40: the clear win.** Vanilla sits at the predict-the-mean
  baseline (0.214), while plastic learns the task (test MSE **0.035**). The
  training curve reached about $3\times10^{-3}$ before an instability near
  epoch 1080 knocked it back up to around 0.1. The final test number
  reflects that partial recovery, so the best it reached was about 10×
  lower. Vanilla also spikes at the end of its run.
- **seq_len ≥ 60: both fail.** Both sit at the baseline (plastic 0.159–0.160,
  vanilla 0.162–0.170) and never leave it in 1200 epochs.

So plasticity roughly doubles the learnable gap compared with vanilla
(vanilla fails at 40, plastic succeeds), but it falls well short of the leaky RNN.
`03_leaky_rnn.ipynb` reached 0.0038 at seq_len=60 and 0.0137 at 100. Its
numbers are last-batch losses rather than held-out MSE, but the gap is far
too large for that to explain.

Section 3 showed the gradient *doesn't* vanish at these lengths, so the
failure isn't a credit-assignment problem. The most likely cause is the
forward pass in Section 2. By $t\approx45$ the self-reinforcing mode in $A$
pushes the network toward saturation, which overwrites the first
frequency before the second window arrives. This is a hypothesis
from the Section 2 figure; it hasn't been tested directly.

## 5. Does the trained network actually rely on plasticity?

A lower loss alone doesn't show that the network uses $A$ as memory. It
could simply have been a better initialization. Ablation test: take each
trained plastic network and, **at test time only**, set $k=0$ (freeze
$A=0$) without changing any trained weights. If the network stores the
first frequency in its synapses, the loss should jump.

In [ ]:
print(f"{'seq_len':>7}  {'plastic':>8}  {'same weights, k=0 at test':>26}")
ablated_test = []
for seq_len, full in zip(seq_lens, plastic_test):
    m = models[seq_len, K]
    m.cell.k.fill_(0.0)
    ablated_test.append(test_mse(m, seq_len))
    m.cell.k.fill_(K)
    print(f'{seq_len:>7}  {full:>8.4f}  {ablated_test[-1]:>26.4f}')

At **seq_len=20 and 40 the network depends on plasticity.** Turning it off
at test time raises the loss 20× (0.0022 → 0.045) and 6× (0.035 → 0.209,
back to the baseline). At seq_len=40 the ablated loss matches the vanilla
model's, which is what you'd expect if $A$ is where the first frequency is
held across the gap.

At seq_len ≥ 60 ablation changes nothing (all ≈ 0.16). Those networks
never learned anything to lose.

## 6. How much plasticity? Sweeping k, and letting gradient descent choose

At `seq_len=100`, sweep $k \in \{0, 1, 2.8, 5\}$ with $p$ fixed at 25. Then
make $k$ a trainable scalar initialized at 2.8 (`learn_k=True`), to see
where the optimizer takes it.

In [ ]:
SWEEP_LEN = 100
k_values = [0.0, 1.0, K, 5.0]
k_test = []
for k in k_values:
    if (SWEEP_LEN, k) not in models:
        models[SWEEP_LEN, k], curves[SWEEP_LEN, k] = train_adding(SWEEP_LEN, k)
    k_test.append(test_mse(models[SWEEP_LEN, k], SWEEP_LEN))
    print(f'k={k:<4}  test MSE={k_test[-1]:.4f}')

learned, learned_curve = train_adding(SWEEP_LEN, K, learn_k=True)
print(f'\nlearnable k: 2.8 -> {learned.cell.k.item():.3f}   test MSE={test_mse(learned, SWEEP_LEN):.4f}')

fig, ax = plt.subplots(figsize=(6, 4))
ax.bar([str(k) for k in k_values], k_test, color=['#c1121f'] + ['#219ebc'] * (len(k_values) - 1))
ax.axhline(2/12, color='gray', ls='--', lw=1, label='predict-the-mean baseline')
ax.set(xlabel='plasticity strength k', ylabel='test MSE', title=f'seq_len={SWEEP_LEN}, p={P:g}')
ax.legend(); fig.tight_layout(); plt.show()

At seq_len=100 **no value of $k$ helps**. All land within 0.157–0.167 of the
baseline ($k=1$ is marginally lowest, at 0.157, which is within
run-to-run noise). Making $k$ learnable doesn't help either: it moved only
from 2.8 to 2.835. Once the model sits at the predict-the-mean solution,
the loss is flat in $k$, so the optimizer has no signal to change it.
Plasticity strength alone isn't the limiting factor at this length.

## 7. Takeaways

| | Vanilla | Leaky (α=0.1)* | Plastic (k=2.8, p=25) |
|---|---|---|---|
| grad norm at lag 60, init | 2e-20 | — | **2e-2** |
| seq_len=20 | **0.0004** | 0.0011 | 0.0022 |
| seq_len=40 | 0.214 | **0.0026** | 0.035 |
| seq_len=60 | 0.162 | **0.0038** | 0.160 |
| seq_len=100 | 0.167 | **0.0137** | 0.159 |
| seq_len=150 | 0.170 | 0.167 | 0.159 |

*\*Leaky numbers from `03_leaky_rnn.ipynb` (last-batch loss, not held-out
MSE). The other two columns are held-out MSE from this notebook.*

- **Hebbian fast weights do act as memory.** At seq_len=40 the plastic
  network solves a task the vanilla one can't, and the ablation shows it
  relies on $A$ to do so.
- **They fix gradient flow completely, but that isn't enough.** The
  backpropagated gradient is flat out to lag 90, yet training still fails
  from seq_len=60. The bottleneck has moved from the backward pass to the
  forward pass.
- **The likely culprit is runaway Hebbian feedback.** With a purely
  Hebbian, non-normalized rule and a bias-driven activity pattern, $A$
  keeps reinforcing one mode and pushes the network toward saturation
  within about 45 steps, whatever the input. It's worth testing directly by
  plotting $A$ in trained networks, or by trying the fixes below.
- **Things worth trying next:**
  - A normalized rule (Oja's rule, or a cap on $\|A\|$) so that $A$
    can't run away.
  - Removing the bias, or giving each unit its own input phase as in the
    paper, so the activity pattern follows the stimulus.
  - A shorter $p$, so old writes decay before they swamp new ones.
  - Letting $k$ go negative (anti-Hebbian).
  - Dynamic nodes as well (leaky units with plastic weights), since the
    leak worked well on this task.